# ResNet-18 on Balanced RGB FFT Spectrograms

This notebook mirrors the **full balanced-split experiment style** of the original
`genai_classifier_resnet18_residual_autocorr.ipynb`, but uses **frequency-domain spectrogram features** instead of residual autocorrelation maps.

Main differences from the small-sample debug notebooks:
- uses the **full manifests**
- builds **balanced train / validation / test datasets** from captions, like the original balanced-split workflow
- reports the same kinds of metrics:
  - train / validation loss
  - train / validation accuracy
  - test loss
  - test accuracy
  - per-class test accuracy
  - confusion matrix

Expected transform source:
- `tools/data_transform_fixed_size.py`

Expected output layout:
- `output/spectrogram_data/metadata/train_spectrogram_manifest.csv`
- `output/spectrogram_data/metadata/validation_spectrogram_manifest.csv`
- `output/spectrogram_data/metadata/test_spectrogram_manifest.csv`

If these manifests do not exist yet, you can generate them by setting `GENERATE_IF_MISSING = True`.


In [ ]:
import csv
import os
import sys
import time
import random
from collections import defaultdict
from pathlib import Path

import matplotlib.pylab as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.models import ResNet18_Weights

try:
    import wandb
except ImportError:
    wandb = None

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print("device =", device)


In [ ]:
# Resolve project paths and import transform helper.

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "prototype-notebooks" else NOTEBOOK_DIR
TOOLS_DIR = PROJECT_ROOT / "tools"
OUTPUT_ROOT = PROJECT_ROOT / "output"
DATA_ROOT = OUTPUT_ROOT / "spectrogram_data"
METADATA_DIR = DATA_ROOT / "metadata"

print("NOTEBOOK_DIR =", NOTEBOOK_DIR)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("TOOLS_DIR =", TOOLS_DIR)
print("DATA_ROOT =", DATA_ROOT)
print("METADATA_DIR =", METADATA_DIR)

if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from data_transform_fixed_size import download_data

WANDB_ENTITY = "william-em-watson-university-of-calgary-in-alberta"
WANDB_PROJECT = "ai-gen-image-detection"
WANDB_API_KEY = os.getenv("WANDB_API_KEY", "")


In [ ]:
# Generate transformed spectrogram data only if needed.
# WARNING: download_data() processes the full split from the starting page onward.

GENERATE_IF_MISSING = True
TARGET_SIZE = 224
DOWNLOAD_PAGE_NUM = 1
DOWNLOAD_PAGE_SIZE = 100
DOWNLOAD_SPLITS = ["train", "validation", "test"]

def manifest_path_for(split_name: str) -> Path:
    return METADATA_DIR / f"{split_name}_spectrogram_manifest.csv"

TRAIN_MANIFEST = manifest_path_for("train")
VAL_MANIFEST = manifest_path_for("validation")
TEST_MANIFEST = manifest_path_for("test")

print("TRAIN_MANIFEST =", TRAIN_MANIFEST, TRAIN_MANIFEST.exists())
print("VAL_MANIFEST =", VAL_MANIFEST, VAL_MANIFEST.exists())
print("TEST_MANIFEST =", TEST_MANIFEST, TEST_MANIFEST.exists())

if GENERATE_IF_MISSING:
    missing = [p for p in [TRAIN_MANIFEST, VAL_MANIFEST, TEST_MANIFEST] if not p.exists()]
    if missing:
        print("Generating missing spectrogram data...")
        for split_name in DOWNLOAD_SPLITS:
            print(f"Generating transformed data for split: {split_name}")
            download_data(
                page_num=DOWNLOAD_PAGE_NUM,
                page_size=DOWNLOAD_PAGE_SIZE,
                split=split_name,
                export_root=DATA_ROOT,
                target_size=TARGET_SIZE,
            )
    else:
        print("All manifests already exist. Skipping generation.")
else:
    print("Generation disabled. Using existing transformed data only.")


In [ ]:
# Manifest loading and balanced-row construction.

def load_manifest_rows(manifest_path: Path):
    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Manifest not found: {manifest_path}\n"
            f"Generate the transformed data first or set the correct DATA_ROOT."
        )
    with manifest_path.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def build_balanced_rows(rows, *, split: str, seed: int):
    rng = random.Random(seed)
    grouped_rows = defaultdict(list)
    for row in rows:
        grouped_rows[row["caption"]].append(row)

    balanced_rows = []
    for caption in sorted(grouped_rows):
        caption_rows = grouped_rows[caption]
        real_rows = [row for row in caption_rows if row["label_a_name"] == "real"]
        ai_rows = [row for row in caption_rows if row["label_a_name"] == "ai_generated"]

        if not real_rows or not ai_rows:
            continue

        selected_ai_row = rng.choice(ai_rows)
        balanced_rows.extend(real_rows)
        balanced_rows.append(selected_ai_row)

    return balanced_rows

SEED = 42

train_rows = load_manifest_rows(TRAIN_MANIFEST)
val_rows = load_manifest_rows(VAL_MANIFEST)
test_rows = load_manifest_rows(TEST_MANIFEST)

train_rows_balanced = build_balanced_rows(train_rows, split="train", seed=SEED)
val_rows_balanced = build_balanced_rows(val_rows, split="validation", seed=SEED)
test_rows_balanced = build_balanced_rows(test_rows, split="test", seed=SEED)

print("Original counts:")
print("  train:", len(train_rows))
print("  validation:", len(val_rows))
print("  test:", len(test_rows))

print("\nBalanced counts:")
print("  train_balanced:", len(train_rows_balanced))
print("  validation_balanced:", len(val_rows_balanced))
print("  test_balanced:", len(test_rows_balanced))

for name, rows in [
    ("train_balanced", train_rows_balanced),
    ("validation_balanced", val_rows_balanced),
    ("test_balanced", test_rows_balanced),
]:
    real_count = sum(r["label_a_name"] == "real" for r in rows)
    ai_count = sum(r["label_a_name"] == "ai_generated" for r in rows)
    print(f"{name}: {real_count} real, {ai_count} ai_generated")


In [ ]:
# Dataset helpers.

class_names = ["real", "ai_generated"]
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

def to_spectrogram_from_npz(sample) -> np.ndarray:
    keys = list(sample.keys())

    if "spectrogram_normalized" in keys:
        return sample["spectrogram_normalized"].astype(np.float32)

    if "spectrogram" in keys:
        spec = sample["spectrogram"].astype(np.float32)
    elif "log_power_spectrum_shifted" in keys:
        spec = sample["log_power_spectrum_shifted"].astype(np.float32)
    elif "power_spectrum_shifted" in keys:
        spec = np.log1p(sample["power_spectrum_shifted"].astype(np.float32))
    elif "magnitude_shifted" in keys:
        spec = np.log1p(sample["magnitude_shifted"].astype(np.float32))
    elif "fft_shifted_real" in keys and "fft_shifted_imag" in keys:
        real = sample["fft_shifted_real"].astype(np.float32)
        imag = sample["fft_shifted_imag"].astype(np.float32)
        spec = np.log1p(real**2 + imag**2)
    elif "fft_real" in keys and "fft_imag" in keys:
        real = sample["fft_real"].astype(np.float32)
        imag = sample["fft_imag"].astype(np.float32)
        spec = np.log1p(real**2 + imag**2)
    else:
        raise KeyError(f"Could not build spectrogram from keys: {keys}")

    mean = spec.mean()
    std = spec.std()
    if std < 1e-6:
        std = 1.0
    return ((spec - mean) / std).astype(np.float32)

class SpectrogramRowsDataset(Dataset):
    def __init__(self, rows, data_root: Path):
        self.rows = rows
        self.data_root = Path(data_root)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        rel_path = row.get("spectrogram_path") or row.get("fft_path") or row.get("image_path")
        if rel_path is None:
            raise KeyError(f"Manifest row missing path field. Row keys: {list(row.keys())}")

        npz_path = self.data_root / rel_path
        if not npz_path.exists():
            raise FileNotFoundError(f"Sample file not found: {npz_path}")

        with np.load(npz_path) as sample:
            feature = to_spectrogram_from_npz(sample)

        tensor = torch.from_numpy(feature).float()
        label = class_to_idx[row["label_a_name"]]
        return tensor, label

batch_size = 16
num_workers = 0

train_dataset = SpectrogramRowsDataset(train_rows_balanced, DATA_ROOT)
val_dataset = SpectrogramRowsDataset(val_rows_balanced, DATA_ROOT)
test_dataset = SpectrogramRowsDataset(test_rows_balanced, DATA_ROOT)

trainloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
valloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
testloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print("Train set:", len(train_dataset))
print("Val set:", len(val_dataset))
print("Test set:", len(test_dataset))
print("First balanced train row:", train_dataset.rows[0] if len(train_dataset) else "EMPTY")


In [ ]:
# Sanity check one batch and visualize one sample.

train_iterator = iter(trainloader)
train_batch = next(train_iterator)

print("Batch tensor shape:", train_batch[0].size())
print("Batch label shape:", train_batch[1].size())

sample_spec = train_batch[0][0].cpu().numpy()
sample_label = class_names[train_batch[1][0].item()]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
channel_titles = ["Red channel", "Green channel", "Blue channel"]

for i, ax in enumerate(axes):
    ax.imshow(sample_spec[i], cmap="magma")
    ax.set_title(channel_titles[i])
    ax.axis("off")

plt.suptitle(f"{sample_label} spectrogram")
plt.tight_layout()
plt.show()


In [ ]:
class AiGenModel(nn.Module):
    def __init__(self, num_classes=2, use_pretrained=False, freeze_backbone=False):
        super().__init__()

        self.use_pretrained = use_pretrained
        self.freeze_backbone = freeze_backbone

        weights = ResNet18_Weights.DEFAULT if use_pretrained else None
        self.feature_extractor = models.resnet18(weights=weights)

        if self.use_pretrained and self.freeze_backbone:
            for name, param in self.feature_extractor.named_parameters():
                if not name.startswith('fc.'):
                    param.requires_grad = False

        in_features = self.feature_extractor.fc.in_features
        self.feature_extractor.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.feature_extractor(x)


In [ ]:
num_classes = len(class_names)
use_pretrained = False
freeze_backbone = False

net = AiGenModel(
    num_classes=num_classes,
    use_pretrained=use_pretrained,
    freeze_backbone=freeze_backbone,
).to(device)
print(net)


In [ ]:
criterion = nn.CrossEntropyLoss()
learning_rate = 1e-3
weight_decay = 1e-4

trainable_params = [param for param in net.parameters() if param.requires_grad]
optimizer = optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

input_representation = 'rgb_fft_log_power_spectrogram_balanced'
feature_key = 'spectrogram_normalized_or_derived'

if WANDB_API_KEY and wandb is not None:
    wandb.login(key=WANDB_API_KEY)
else:
    print('W&B login skipped. Set WANDB_API_KEY and install wandb if you want remote logging.')

wandb_run = None
if wandb is not None:
    wandb_config = {
        'train_manifest': str(TRAIN_MANIFEST),
        'val_manifest': str(VAL_MANIFEST),
        'test_manifest': str(TEST_MANIFEST),
        'data_root': str(DATA_ROOT),
        'model_name': 'resnet18',
        'use_pretrained': use_pretrained,
        'freeze_backbone': freeze_backbone,
        'input_representation': input_representation,
        'feature_key': feature_key,
        'batch_size': batch_size,
        'num_workers': num_workers,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'num_classes': num_classes,
        'device': str(device),
        'seed': SEED,
    }

    wandb_run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        config=wandb_config,
        mode='online' if WANDB_API_KEY else 'disabled',
    )

    wandb.define_metric('epoch')
    wandb.define_metric('train_loss', step_metric='epoch')
    wandb.define_metric('train_acc', step_metric='epoch')
    wandb.define_metric('val_loss', step_metric='epoch')
    wandb.define_metric('val_acc', step_metric='epoch')


In [ ]:
# Match the original residual/autocorr notebook settings more closely.

nepochs = 10
PATH = './genai_resnet18_spectrogram_balanced_best.pth'

if wandb_run is not None:
    wandb.config.update({'nepochs': nepochs}, allow_val_change=True)

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(nepochs):
    epoch_start_time = time.perf_counter()
    net.train()
    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for inputs, labels in trainloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / max(len(train_dataset), 1)
    train_acc = train_correct / max(train_total, 1)

    net.eval()
    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in valloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = net(inputs)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / max(len(val_dataset), 1)
    val_acc = val_correct / max(val_total, 1)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    epoch_time_sec = time.perf_counter() - epoch_start_time
    current_lr = optimizer.param_groups[0]['lr']

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(net.state_dict(), PATH)

    log_payload = {
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'lr': current_lr,
        'epoch_time_sec': epoch_time_sec,
    }

    if wandb_run is not None:
        wandb.log(log_payload)

    print(
        f"Epoch {epoch + 1}/{nepochs} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={current_lr:.6f} time={epoch_time_sec:.2f}s"
    )


In [ ]:
# Load best model before test evaluation.

net = AiGenModel(
    num_classes=len(class_names),
    use_pretrained=use_pretrained,
    freeze_backbone=freeze_backbone,
).to(device)
net.load_state_dict(torch.load(PATH, map_location=device))
net.eval()

correct = 0
total = 0
test_loss = 0.0

test_start_time = time.perf_counter()
with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_inference_time_sec = time.perf_counter() - test_start_time
test_loss = test_loss / max(len(test_dataset), 1)
test_accuracy = correct / max(total, 1)

test_metrics = {
    'test_loss': test_loss,
    'test_accuracy': test_accuracy,
    'test_accuracy_percent': 100 * test_accuracy,
    'test_inference_time_sec': test_inference_time_sec,
}

if wandb_run is not None:
    wandb.log(test_metrics)
    for key, value in test_metrics.items():
        wandb.run.summary[key] = value

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {100 * test_accuracy:.2f}%")
print(f"Test inference time: {test_inference_time_sec:.2f}s")


In [ ]:
# Per-class test accuracy.

class_correct = {name: 0 for name in class_names}
class_total = {name: 0 for name in class_names}

with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)

        for label, pred in zip(labels, predicted):
            class_name = class_names[label.item()]
            class_total[class_name] += 1
            if label.item() == pred.item():
                class_correct[class_name] += 1

for class_name in class_names:
    accuracy = class_correct[class_name] / max(class_total[class_name], 1)
    print(f'{class_name}: {100 * accuracy:.2f}% ({class_correct[class_name]}/{class_total[class_name]})')
    if wandb_run is not None:
        wandb.log({f'test_accuracy_{class_name}': accuracy})
        wandb.run.summary[f'test_accuracy_{class_name}'] = accuracy


In [ ]:
# Training curves.

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.title('Loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='train')
plt.plot(history['val_acc'], label='val')
plt.title('Accuracy')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Test confusion matrix.

confusion_matrix = torch.zeros((len(class_names), len(class_names)), dtype=torch.int64)

with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)

        for label, pred in zip(labels.view(-1), predicted.view(-1)):
            confusion_matrix[label.item(), pred.item()] += 1

cm = confusion_matrix.cpu().numpy()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
fig.colorbar(im, ax=ax)

ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Test Confusion Matrix')

threshold = cm.max() / 2 if cm.size else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            cm[i, j],
            ha='center',
            va='center',
            color='white' if cm[i, j] > threshold else 'black',
        )

plt.tight_layout()
plt.show()

if wandb_run is not None:
    wandb.finish()
